# Project 1 — Issue Report Classification
## Notebook 05: Final comparison and error analysis

Pulls every saved result into one table, compares against the published SetFit
baseline, and analyses where the best model fails.

All results are read from `results/tables/*.json`, written by
`scripts/run_experiments.py`. Nothing is re-trained here, so the report can be
rebuilt without a GPU and the numbers cannot drift from what the code
produced.

In [ ]:
import json

import pandas as pd

from ai4se import error_analysis as ea
from ai4se.classical import TUNED_PREPROCESSING, tuned_logistic_regression
from ai4se.evaluation import SETFIT_OVERALL, leaderboard_from_disk, to_latex
from ai4se.loader import PROJECT_ROOT, load_split
from ai4se.preprocessing import make_cleaner

pd.set_option("display.width", 160)
TABLES = PROJECT_ROOT / "results" / "tables"

board = leaderboard_from_disk(TABLES)
board

---
## 1. Reading the leaderboard

| Group | Best result | Gap to baseline |
|---|---|---|
| Trivial floors | 0.5395 (keyword rules) | −0.288 |
| Classical ML (Track B) | 0.7603 | −0.067 |
| Neural (Track C) | 0.7438 | −0.083 |
| Frozen embeddings | 0.7557 (MPNet) | −0.071 |
| SetFit reproduction | 0.8102 (MPNet) | −0.017 |
| **Ensembles** | **0.8168** | **−0.010** |
| **SetFit (published baseline)** | **0.8270** | — |

The best result, a soft-voting ensemble of SetFit, TF-IDF and frozen MPNet,
is 1.02 points below the published baseline — under the 3.9-point difference a
conservative test can detect against it, so it is **statistically indistinguishable from the
baseline**.

Two caveats frame everything below. First, a model reading **only the creation
timestamp** scores 0.7071, because no bug report predates 2021 — so the
meaningful floor is ~0.70, not the majority class's 0.17. Second, how small a
difference is detectable depends on the pair of models: 1.8–3.1 points overall
(3.9–6.9 per project) for real pairs, and 3.9 overall (7.6–9.8 per project)
against the published baseline, whose per-issue predictions are unavailable.

### The encoder effect, measured three times

| Stage | Conclusion | Status |
|---|---|---|
| Projected additively | SetFit-MPNet ≈ 0.8482, beats baseline | **wrong by 0.038** |
| Measured, uncontrolled | encoder worth +0.0120 after fine-tuning | **confounded** |
| Measured, matched settings | encoder worth **+0.0286** | holds |

The uncontrolled comparison ran MiniLM at `batch_size=16, max_seq_length=256`
and MPNet at 8 and 128 — MPNet was handicapped. Re-running MiniLM at MPNet's
settings gives 0.7816, so **the reduced settings alone cost 0.0166**.

| | MiniLM | MPNet | encoder effect |
|---|---|---|---|
| Frozen + LogReg | 0.7057 | 0.7557 | **+0.0500** |
| SetFit @ batch 8, seq 128 | 0.7816 | 0.8102 | **+0.0286** |

Sub-additivity is real but mild: the encoder keeps **57%** of its frozen value
after fine-tuning, not the 24% the uncontrolled numbers implied.

Two methodological points, in order of importance:

1. Effects measured separately were assumed to compose, and they did not.
2. The first correction was itself overstated, because the comparison behind it
   was confounded. Only the matched re-run settled it.

**Training settings matter about as much as encoder size here** — halving batch
and sequence length cost 0.0166 against +0.0286 for doubling encoder depth and
width. A result quoted without them is not comparable with another.

In [ ]:
repos = ["react", "tensorflow", "vscode", "bitcoin", "opencv"]
display(board.loc[:, repos].describe().loc[["mean", "std", "min", "max"]].round(4))
print("\nhardest and easiest project for each model (top 6):")
subset = board.loc[:, repos].head(6)
display(pd.DataFrame({
    "easiest": subset.idxmax(axis=1),
    "hardest": subset.idxmin(axis=1),
    "spread": (subset.max(axis=1) - subset.min(axis=1)).round(4),
}))

`bitcoin/bitcoin` is hardest for almost every model, and for the published
baseline too (0.7555, its lowest). Project difficulty is a property of the data
rather than of any approach.

The ensembles score 0.7691 and 0.7694 on `bitcoin` against the baseline's
0.7555, and SetFit on MPNet scores 0.8710 on `tensorflow` against 0.8644. These
are **numerically** above the baseline but are 0.7–1.4 points on 300 items,
far below the 7.8–9.8-point per-project threshold against the published
baseline. They are ties, not
wins, and should not be reported as beating the baseline.

In [ ]:
train = load_split("train", kind="memory")
test = load_split("test", kind="memory")
train.apply(make_cleaner(**TUNED_PREPROCESSING))
test.apply(make_cleaner(**TUNED_PREPROCESSING))

predictions = ea.collect_predictions(tuned_logistic_regression, train, test)
display(ea.per_repository_scores(predictions))

In [ ]:
ea.confusion_summary(predictions)

**`bug` misclassified as `question` is the single biggest error type**,
accounting for over a quarter of all mistakes. The two classes genuinely
overlap: a user who is unsure whether behaviour is broken writes something that
reads as both a bug report and a question, and the GitHub label they chose
reflects a maintainer's judgement rather than anything in the text.

The three `question` confusions together account for a large share of errors,
which is consistent with `question` having the lowest per-class F1 throughout.

### Does length matter?

In [ ]:
ea.errors_by_length(predictions, bins=5)

Accuracy peaks in the middle of the length distribution and falls at both ends.
Very short issues carry too little text to classify; very long ones are diluted,
and truncation discards part of them. This justifies reporting the truncation
threshold as a hyperparameter rather than an implementation detail.

### What misleads the model?

In [ ]:
display(ea.misleading_terms(predictions, "question", "bug", n=10))
display(ea.misleading_terms(predictions, "bug", "question", n=10))

The `question → bug` table is dominated by fragments of OpenCV's issue
template. Project-specific boilerplate is correlated with the class in the
training data, so the model learns the template rather than the content. This is
a concrete argument for more aggressive template stripping — and a reminder that
the per-project protocol lets each classifier overfit to its own project's
conventions.

### The most confident mistakes

Confident errors are the informative ones. A near-tie means a genuinely
ambiguous issue; a confident mistake means the model learned something wrong, or
the ground-truth label is questionable.

In [ ]:
mistakes = ea.worst_mistakes(predictions, n=12)
ea.mistakes_frame(mistakes)[["repository", "true", "predicted", "confidence", "title"]]

Rather than eyeball these, quantify what is going on. Many titles carry an
explicit self-declaration of their type — `[Feature Request]`, `How to ...`,
`[Bug]`, `... fails` — which can be checked against the ground-truth label.

In [ ]:
from collections import Counter

worst = json.loads((TABLES / "error_analysis.json").read_text())["worst"]

markers = {
    "feature": ["feature request", "[feature]", "allow ", "add support", "please add"],
    "bug": ["[bug]", "bug:", "crash", "error", "fails", "broken"],
    "question": ["how to", "how do", "what is", "why "],
}

conflicts = []
for mistake in worst:
    title = mistake["title"].lower()
    for declared, patterns in markers.items():
        if declared != mistake["true"] and any(p in title for p in patterns):
            conflicts.append({**mistake, "title_declares": declared})
            break

print("true label of the 25 most confident mistakes:")
for label, count in Counter(m["true"] for m in worst).most_common():
    print(f"  {label:<10}{count}")

agreed = sum(1 for c in conflicts if c["predicted"] == c["title_declares"])
print(f"\n{len(conflicts)}/{len(worst)} confident errors have a title that declares "
      f"a different class than the label")
print(f"of those, the model agreed with the title in {agreed} cases")

pd.DataFrame(conflicts)[["true", "title_declares", "predicted", "title"]]

Two things fall out of this.

**`question` is where the model breaks down.** Fourteen of the twenty-five most
confident errors have `question` as their true label, against six for `feature`
and five for `bug`. That matches its consistently lowest per-class F1.

**About a third of confident errors look like label noise.** Nine of the
twenty-five have a title that explicitly declares a different type than the
ground-truth label — an issue titled `[Feature Request] Add workbench action to
split editor terminal below` is labelled `question` — and in most of those the
model's prediction agrees with the title rather than the label.

These labels come from maintainers applying project conventions, not from an
annotation protocol with adjudication. A ceiling below 1.0 is therefore built
into the dataset, and part of the remaining gap to any model is unreachable.
This belongs in threats to validity, and it is worth noting that the published
baseline faces exactly the same ceiling.

One further observation from scanning the list: at least one confident error is
an issue written in Portuguese. The dataset is not language-filtered, but this
is a marginal effect — one case in twenty-five — not a major error source.

In [ ]:
latex = to_latex(
    board,
    caption=(
        "Per-repository and cross-repository $F_1$ for every model, "
        "against the NLBSE'24 SetFit baseline."
    ),
    label="final-leaderboard",
    path=TABLES / "final_leaderboard.tex",
)
print(latex[:600], "...")

---
## 4. Summary

| Finding | Evidence |
|---|---|
| The benchmark has a temporal confound | a timestamp-only model scores 0.7071; no bug predates 2021 |
| Detectable difference depends on the model pair | 1.8–3.1 points overall, 3.9 vs the published baseline |
| Best result 0.8168 is indistinguishable from the baseline | 1.02 points below it, under the 3.9-point threshold |
| Contrastive fine-tuning is worth +0.0759 | matched settings, MiniLM 0.7816 vs frozen 0.7057 |
| A larger encoder is worth +0.0286 | matched settings, MPNet 0.8102 vs MiniLM 0.7816 |
| Encoder and fine-tuning are sub-additive | encoder keeps 57% of its frozen +0.0500 |
| Training settings matter as much as encoder size | reduced batch/seq cost 0.0166 |
| An additive projection failed by 0.038 | predicted 0.8482, measured 0.8102 |
| Ensembling beats every individual model | +0.0066 over its best member; best AUC 0.9319 |
| Neural and SetFit runs vary by ~0.01–0.015 | CNN moved 0.0140 across environments |
| Aggressive cleaning hurts | `full` scores 0.015 below `light`, consistently |
| Per-project training beats global | +0.068 on all five, with a fifth of the data |
| `bug` vs `question` is the dominant confusion | >25% of all errors |
| ~⅓ of confident errors look like label noise | title declares a different type than the label |

### Threats to validity worth carrying into the report

Every result was regenerated from a clean checkout on one machine.
**Twenty-one of twenty-two models returned bit-identical scores.**

The exception is informative rather than worrying. `SetFit (MPNet)` moved its
cross-repository F1 by 0.00004, while its per-repository scores moved by up to
0.0099 — `vscode` fell 0.0099 and `opencv` rose 0.0095, largely cancelling.
**The competition's averaged metric is roughly an order of magnitude more
stable than any single project's figure**, because it averages five
independent classifiers. Quote per-repository numbers to show *where* models
differ, not to claim one beats another by a small margin on one project.

Across *devices* the picture is different: the CNN scores 0.7578 on a CPU and
0.7438 on a GPU from identical code and seed, because the two backends use
different kernels and accumulation orders. Report a neural result with the
device that produced it.

Two earlier claims in this analysis did not survive being checked — a cuDNN
nondeterminism diagnosis whose fix changed nothing, and a SetFit variance
estimate that turned out to span a code change. Both are documented in
`code_overview.md` §7.3.